# 01 轨迹转城市接续

写入日期：2026-10-16  
修改日期：2026-10-17  

本次修改：把城市为空的5行记录当作噪音删除，并单独保存删除记录。删除后，其余记录继续按时间排列。核心代码仍全部放在本Notebook中。

## 说明

本步骤读取 `Data/00_原始数据_只读` 中三份GNSS文件，按年份分别整理每台农机先后到过的城市。

处理城市为空的记录时采用以下简单规则：

1. 原始文件不改动。
2. 生成接续数据前删除城市为空的5行记录。
3. 删除后，其余城市继续按时间排列，因此被删除记录前后的有效城市可能重新连接。
4. 被删除的5行另存到 `Results/00_数据审计/城市缺失删除记录.csv`，可以查到来源文件和原始行号。

如果农机编号、日期、开始时间或结束时间无法读取，程序会停止，因为这类问题会导致记录无法正确排序。

连续出现的同一城市合并为一次访问；离开后再次回到同一城市仍保留。`edge_level`表示两个城市位置相隔几步，`edge_weight`按距离逐级减半。输出只表示先后关系，不能证明因果关系。

## 参数

In [1]:
"""分年生成农机城市候选接续边，并输出完整质量审计。

正式口径以项目 PLAN.md 与 DECISIONS.md 为准。该模块不修改原始数据，
也不把缺失值填为 0；所有正式输出在全部年份通过检查后才从临时目录移入。
"""

from __future__ import annotations

import csv
import hashlib
import json
import os
import shutil
import tempfile
from datetime import datetime
from pathlib import Path
from typing import Any
from zoneinfo import ZoneInfo

import duckdb


WRITE_DATE = "2026-10-16"
MODIFY_DATE = "2026-10-17"
DECAY_BASE = 0.5
EXPECTED_COLUMNS = {
    "info_id",
    "id",
    "did",
    "date",
    "start_time",
    "end_time",
    "center_time",
    "center_location",
    "tag",
}
YEAR_CONFIG = {
    2021: {
        "input": "Data/00_原始数据_只读/final_filtered_21_modified.csv",
        "output": "Data/03_有向边数据/2021年农机城市接续.csv",
        "expected_rows": 770_845,
        "start_date": "2021-05-01",
        "end_date": "2021-06-25",
        "expected_missing_city": 1,
    },
    2022: {
        "input": "Data/00_原始数据_只读/final_filtered_22_modified.csv",
        "output": "Data/03_有向边数据/2022年农机城市接续.csv",
        "expected_rows": 2_475_805,
        "start_date": "2022-05-01",
        "end_date": "2022-06-25",
        "expected_missing_city": 0,
    },
    2023: {
        "input": "Data/00_原始数据_只读/final_filtered_23_modified.csv",
        "output": "Data/03_有向边数据/2023年农机城市接续.csv",
        "expected_rows": 3_976_426,
        "start_date": "2023-05-01",
        "end_date": "2023-06-25",
        "expected_missing_city": 4,
    },
}
AUDIT_RELATIVE_PATH = "Results/00_数据审计/城市接续审计.csv"
DELETION_RELATIVE_PATH = "Results/00_数据审计/城市缺失删除记录.csv"
DELETION_COLUMNS = [
    "年份", "来源文件", "源数据行号", "农机编号",
    "作业日期", "开始时间", "结束时间", "删除原因",
]
SORT_SQL = (
    "date_value, start_ts, end_ts, center_ts NULLS LAST, "
    "info_id NULLS LAST, id NULLS LAST, source_row_no"
)
OUTPUT_COLUMNS = [
    "边记录编号",
    "年份",
    "农机编号",
    "链段编号",
    "起点访问序号",
    "终点访问序号",
    "起点城市",
    "终点城市",
    "边级",
    "跨越城市数",
    "衰减权重",
    "起点进入时间",
    "起点离开时间",
    "终点进入时间",
    "终点离开时间",
    "时间间隔小时",
    "起点原始记录数",
    "终点原始记录数",
    "起点首行号",
    "起点末行号",
    "终点首行号",
    "终点末行号",
    "起点是否回访",
    "终点是否回访",
    "终点城市访问序次",
    "是否直接回流边",
    "是否同城端点",
    "是否时间冲突",
    "是否同起点时间",
    "来源文件",
]

def find_project_root(start_path: Path) -> Path:
    """从当前工作目录向上查找项目根目录，避免依赖手工路径。"""
    for candidate in (start_path.resolve(), *start_path.resolve().parents):
        if (candidate / "README.md").is_file() and (candidate / "Data").is_dir():
            return candidate
    raise FileNotFoundError("无法从当前工作目录定位包含 README.md 与 Data 的项目根目录。")


PROJECT_ROOT = find_project_root(Path.cwd())
NOTEBOOK_PATH = PROJECT_ROOT / "Code" / "01_轨迹转城市接续" / "build_machine_city_edges.ipynb"
ALLOW_OVERWRITE = os.environ.get("CITY_EDGE_ALLOW_OVERWRITE", "0") == "1"

print(
    {
        "project_root": str(PROJECT_ROOT),
        "notebook": str(NOTEBOOK_PATH),
        "decay_base": DECAY_BASE,
        "allow_overwrite": ALLOW_OVERWRITE,
    }
)

{'project_root': 'C:\\Users\\Administrator\\Desktop\\W', 'notebook': 'C:\\Users\\Administrator\\Desktop\\W\\Code\\01_轨迹转城市接续\\build_machine_city_edges.ipynb', 'decay_base': 0.5, 'allow_overwrite': True}


## 读取

In [2]:
def _now_text() -> str:
    return datetime.now(ZoneInfo("Asia/Shanghai")).isoformat(timespec="seconds")


def _sql_path(path: Path) -> str:
    return path.resolve().as_posix().replace("'", "''")


def _sha256(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def _read_header(path: Path) -> list[str]:
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        return next(csv.reader(handle))


def _scalar(connection: duckdb.DuckDBPyConnection, sql: str) -> Any:
    return connection.execute(sql).fetchone()[0]


def _assert_zero(connection: duckdb.DuckDBPyConnection, sql: str, message: str) -> None:
    value = int(_scalar(connection, sql) or 0)
    if value != 0:
        raise RuntimeError(f"{message}（发现 {value:,} 条）")


def _copy_with_bom(source: Path, destination: Path) -> None:
    with source.open("rb") as src, destination.open("wb") as dst:
        dst.write(b"\xef\xbb\xbf")
        shutil.copyfileobj(src, dst, length=8 * 1024 * 1024)


def _audit_row(
    rows: list[dict[str, Any]],
    run_id: str,
    year: int | str,
    group: str,
    metric: str,
    value: Any,
    unit: str,
    status: str,
    note: str,
    input_file: str,
    output_file: str,
) -> None:
    rows.append(
        {
            "运行编号": run_id,
            "运行时间": _now_text(),
            "年份": year,
            "指标组": group,
            "指标": metric,
            "数值": value,
            "单位": unit,
            "检查状态": status,
            "说明": note,
            "输入文件": input_file,
            "输出文件": output_file,
        }
    )


def _sequence_edges(sequence: list[str | None]) -> list[tuple[str, str, int, float]]:
    """只用于小样测试；None表示这行会先被删除。"""
    kept_visits: list[str] = []
    for city in sequence:
        if city is None:
            continue
        if not kept_visits or kept_visits[-1] != city:
            kept_visits.append(city)

    result: list[tuple[str, str, int, float]] = []
    for source_index in range(len(kept_visits)):
        for target_index in range(source_index + 1, len(kept_visits)):
            level = target_index - source_index
            result.append(
                (
                    kept_visits[source_index],
                    kept_visits[target_index],
                    level,
                    DECAY_BASE ** (level - 1),
                )
            )
    return result


def run_unit_tests() -> None:
    """检查删除空城市、合并连续同城、边级、权重和回访。"""
    abcd = _sequence_edges(["A", "B", "C", "D"])
    assert len(abcd) == 6 and abs(sum(item[3] for item in abcd) - 4.25) < 1e-15

    repeated_city = _sequence_edges(["A", "A", "B", "A", "A"])
    assert len(repeated_city) == 3 and ("A", "A", 2, 0.5) in repeated_city

    deleted_noise = _sequence_edges(["A", "B", None, "C", "D"])
    assert deleted_noise == abcd

    return_path = _sequence_edges(["A", "B", "C", "D", "C", "E"])
    assert len(return_path) == 15
    assert abs(sum(item[3] for item in return_path) - 8.0625) < 1e-15

    repeated_pairs = _sequence_edges(["A", "B", "A", "B"])
    assert len(repeated_pairs) == 6


In [3]:
def inspect_inputs(project_root: Path) -> list[dict[str, Any]]:
    """只读检查输入路径、表头、大小和预期输出位置。"""
    project_root = project_root.resolve()
    report: list[dict[str, Any]] = []
    for year, config in YEAR_CONFIG.items():
        input_path = project_root / config["input"]
        output_path = project_root / config["output"]
        if not input_path.is_file():
            raise FileNotFoundError(f"缺少正式输入：{input_path}")
        header = _read_header(input_path)
        missing = sorted(EXPECTED_COLUMNS.difference(header))
        if missing:
            raise RuntimeError(f"{year} 年输入缺少字段：{missing}")
        report.append(
            {
                "year": year,
                "input": str(input_path),
                "input_bytes": input_path.stat().st_size,
                "column_count": len(header),
                "output": str(output_path),
            }
        )
    return report


def _prepare_overwrite_backup(project_root: Path, paths: list[Path]) -> str:
    existing = [path for path in paths if path.exists()]
    if not existing:
        return ""
    timestamp = datetime.now(ZoneInfo("Asia/Shanghai")).strftime("%Y-%m-%d_%H%M%S")
    backup_dir = (project_root / "Archive" / f"{timestamp}_城市接续重跑前").resolve()
    if project_root.resolve() not in backup_dir.parents:
        raise RuntimeError("备份目录越出项目范围，已停止。")
    backup_dir.mkdir(parents=True, exist_ok=False)
    for path in existing:
        relative = path.resolve().relative_to(project_root.resolve())
        safe_name = "__".join(relative.parts)
        shutil.copy2(path, backup_dir / safe_name)
    return str(backup_dir.relative_to(project_root)).replace("\\", "/")

In [4]:
input_inventory = inspect_inputs(PROJECT_ROOT)
print(f"已确认 {len(input_inventory)} 份正式输入。")
input_inventory

已确认 3 份正式输入。


[{'year': 2021,
  'input': 'C:\\Users\\Administrator\\Desktop\\W\\Data\\00_原始数据_只读\\final_filtered_21_modified.csv',
  'input_bytes': 200790542,
  'column_count': 26,
  'output': 'C:\\Users\\Administrator\\Desktop\\W\\Data\\03_有向边数据\\2021年农机城市接续.csv'},
 {'year': 2022,
  'input': 'C:\\Users\\Administrator\\Desktop\\W\\Data\\00_原始数据_只读\\final_filtered_22_modified.csv',
  'input_bytes': 649368633,
  'column_count': 26,
  'output': 'C:\\Users\\Administrator\\Desktop\\W\\Data\\03_有向边数据\\2022年农机城市接续.csv'},
 {'year': 2023,
  'input': 'C:\\Users\\Administrator\\Desktop\\W\\Data\\00_原始数据_只读\\final_filtered_23_modified.csv',
  'input_bytes': 1075228346,
  'column_count': 26,
  'output': 'C:\\Users\\Administrator\\Desktop\\W\\Data\\03_有向边数据\\2023年农机城市接续.csv'}]

## 处理

In [5]:
def _process_year(
    project_root: Path,
    temp_dir: Path,
    year: int,
    config: dict[str, Any],
    run_id: str,
    audit_rows: list[dict[str, Any]],
    deletion_rows: list[dict[str, Any]],
) -> dict[str, Any]:
    input_path = (project_root / config["input"]).resolve()
    output_path = (project_root / config["output"]).resolve()
    input_rel = config["input"]
    output_rel = config["output"]
    database_path = temp_dir / f"{year}.duckdb"
    csv_without_bom = temp_dir / f"{year}_utf8.csv"
    csv_with_bom = temp_dir / output_path.name
    connection = duckdb.connect(str(database_path))
    connection.execute(f"PRAGMA threads={max(1, min(os.cpu_count() or 1, 8))}")
    connection.execute("PRAGMA preserve_insertion_order=true")
    try:
        connection.execute(
            f"""
            CREATE TABLE raw AS
            SELECT
                row_number() OVER ()::BIGINT AS source_row_no,
                trim(info_id) AS info_id,
                trim(id) AS id,
                trim(did) AS did,
                trim(date) AS date_text,
                trim(start_time) AS start_time_text,
                trim(end_time) AS end_time_text,
                trim(center_time) AS center_time_text,
                trim(center_location) AS center_location_text,
                trim(tag) AS tag_text
            FROM read_csv_auto(
                '{_sql_path(input_path)}',
                header=true,
                all_varchar=true,
                parallel=false,
                ignore_errors=false
            )
            """
        )
        connection.execute(
            """
            CREATE TABLE parsed AS
            SELECT
                source_row_no,
                NULLIF(info_id, '') AS info_id,
                NULLIF(id, '') AS id,
                NULLIF(did, '') AS did,
                NULLIF(date_text, '') AS date_text,
                try_cast(NULLIF(date_text, '') AS DATE) AS date_value,
                NULLIF(start_time_text, '') AS start_time_text,
                try_cast(NULLIF(start_time_text, '') AS TIMESTAMP) AS start_ts,
                NULLIF(end_time_text, '') AS end_time_text,
                try_cast(NULLIF(end_time_text, '') AS TIMESTAMP) AS end_ts,
                NULLIF(center_time_text, '') AS center_time_text,
                coalesce(
                    try_strptime(NULLIF(center_time_text, ''), '%Y%m%d%H%M%S'),
                    try_cast(NULLIF(center_time_text, '') AS TIMESTAMP)
                ) AS center_ts,
                NULLIF(center_location_text, '') AS city,
                tag_text
            FROM raw
            """
        )

        input_rows = int(_scalar(connection, "SELECT count(*) FROM parsed"))
        unique_did = int(_scalar(connection, "SELECT count(DISTINCT did) FROM parsed"))
        city_count = int(_scalar(connection, "SELECT count(DISTINCT city) FROM parsed"))
        date_min, date_max = connection.execute(
            "SELECT min(date_value)::VARCHAR, max(date_value)::VARCHAR FROM parsed"
        ).fetchone()
        critical_counts = connection.execute(
            """
            SELECT
                count(*) FILTER (WHERE did IS NULL),
                count(*) FILTER (WHERE date_text IS NULL OR date_value IS NULL),
                count(*) FILTER (WHERE start_time_text IS NULL OR start_ts IS NULL),
                count(*) FILTER (WHERE end_time_text IS NULL OR end_ts IS NULL),
                count(*) FILTER (WHERE city IS NULL),
                count(*) FILTER (WHERE start_ts IS NOT NULL AND end_ts IS NOT NULL AND end_ts < start_ts),
                count(*) FILTER (WHERE tag_text IS DISTINCT FROM '1'),
                count(*) FILTER (WHERE date_value IS NOT NULL AND start_ts IS NOT NULL
                                  AND date_value IS DISTINCT FROM cast(start_ts AS DATE))
            FROM parsed
            """
        ).fetchone()
        (
            missing_did,
            invalid_date,
            invalid_start,
            invalid_end,
            missing_city,
            reversed_time,
            non_field_tag,
            date_time_mismatch,
        ) = map(int, critical_counts)

        if input_rows != config["expected_rows"]:
            raise RuntimeError(
                f"{year} 年输入行数为 {input_rows:,}，与正式字典 {config['expected_rows']:,} 不一致。"
            )
        if date_min != config["start_date"] or date_max != config["end_date"]:
            raise RuntimeError(
                f"{year} 年日期范围为 {date_min}—{date_max}，与正式范围不一致。"
            )
        unplaceable = missing_did + invalid_date + invalid_start + invalid_end + reversed_time
        if unplaceable:
            raise RuntimeError(
                f"{year} 年有 {unplaceable:,} 条记录无法可靠定位或排序："
                f"did缺失={missing_did}, 日期无效={invalid_date}, 开始时间无效={invalid_start}, "
                f"结束时间无效={invalid_end}, 结束早于开始={reversed_time}。已停止，未写正式输出。"
            )
        if non_field_tag:
            raise RuntimeError(f"{year} 年存在 {non_field_tag:,} 条 tag 不为 1 的记录。")
        if missing_city != config["expected_missing_city"]:
            raise RuntimeError(
                f"{year} 年城市为空的记录数为 {missing_city:,}，"
                f"与已确认数量 {config['expected_missing_city']:,} 不一致，已停止。"
            )

        same_start_diff_city_groups = int(
            _scalar(
                connection,
                """
                SELECT count(*) FROM (
                    SELECT did, start_ts
                    FROM parsed
                    GROUP BY did, start_ts
                    HAVING count(DISTINCT city) > 1
                )
                """,
            )
        )
        exact_sort_tie_groups = int(
            _scalar(
                connection,
                """
                SELECT count(*) FROM (
                    SELECT did, date_value, start_ts, end_ts, center_ts, info_id, id
                    FROM parsed
                    GROUP BY ALL
                    HAVING count(*) > 1
                )
                """,
            )
        )

        deleted_values = connection.execute(
            """
            SELECT source_row_no, did, date_text, start_time_text, end_time_text
            FROM parsed
            WHERE city IS NULL
            ORDER BY source_row_no
            """
        ).fetchall()
        for source_row_no, did_value, date_text, start_text, end_text in deleted_values:
            deletion_rows.append(
                {
                    "年份": year,
                    "来源文件": input_rel,
                    "源数据行号": source_row_no,
                    "农机编号": did_value,
                    "作业日期": date_text,
                    "开始时间": start_text,
                    "结束时间": end_text,
                    "删除原因": "城市为空，按项目决定作为噪音删除",
                }
            )

        connection.execute(
            """
            CREATE TABLE valid_records AS
            SELECT *, 1::INTEGER AS chain_segment
            FROM parsed
            WHERE city IS NOT NULL
            """
        )
        connection.execute(
            f"""
            CREATE TABLE visit_flags AS
            SELECT *,
                CASE
                    WHEN lag(city) OVER (
                        PARTITION BY did, chain_segment ORDER BY {SORT_SQL}
                    ) IS NULL THEN 1
                    WHEN city IS DISTINCT FROM lag(city) OVER (
                        PARTITION BY did, chain_segment ORDER BY {SORT_SQL}
                    ) THEN 1
                    ELSE 0
                END AS new_visit
            FROM valid_records
            """
        )
        connection.execute(
            f"""
            CREATE TABLE visit_rows AS
            SELECT *,
                sum(new_visit) OVER (
                    PARTITION BY did, chain_segment
                    ORDER BY {SORT_SQL} ROWS UNBOUNDED PRECEDING
                )::BIGINT AS visit_group
            FROM visit_flags
            """
        )
        connection.execute(
            f"""
            CREATE TABLE visits_base AS
            SELECT
                {year}::INTEGER AS year,
                did,
                chain_segment,
                visit_group,
                first(city ORDER BY {SORT_SQL}) AS city,
                min(start_ts) AS entry_time,
                max(end_ts) AS exit_time,
                count(*)::BIGINT AS raw_record_count,
                first(source_row_no ORDER BY {SORT_SQL})::BIGINT AS first_source_row_no,
                last(source_row_no ORDER BY {SORT_SQL})::BIGINT AS last_source_row_no
            FROM visit_rows
            GROUP BY did, chain_segment, visit_group
            """
        )
        connection.execute(
            """
            CREATE TABLE visits AS
            SELECT *,
                row_number() OVER (
                    PARTITION BY did, chain_segment ORDER BY visit_group
                )::INTEGER AS visit_order,
                row_number() OVER (
                    PARTITION BY did, chain_segment, city ORDER BY visit_group
                )::INTEGER AS city_visit_no
            FROM visits_base
            """
        )
        connection.execute(
            """
            CREATE TABLE chain_stats AS
            SELECT year, did, chain_segment, count(*)::BIGINT AS visit_count
            FROM visits
            GROUP BY year, did, chain_segment
            """
        )

        valid_records = int(_scalar(connection, "SELECT count(*) FROM valid_records"))
        chain_segments = int(_scalar(connection, "SELECT count(*) FROM chain_stats"))
        visit_count = int(_scalar(connection, "SELECT count(*) FROM visits"))
        max_visits = int(_scalar(connection, "SELECT max(visit_count) FROM chain_stats") or 0)
        expected_edges = int(
            _scalar(connection, "SELECT sum(visit_count * (visit_count - 1) / 2)::BIGINT FROM chain_stats") or 0
        )
        expected_level_one = int(
            _scalar(connection, "SELECT sum(greatest(visit_count - 1, 0))::BIGINT FROM chain_stats") or 0
        )
        expected_weight = float(
            _scalar(
                connection,
                "SELECT sum(2 * visit_count - 4 + pow(2, 2 - visit_count)) FROM chain_stats",
            )
            or 0.0
        )
        _assert_zero(
            connection,
            f"SELECT abs((SELECT sum(raw_record_count) FROM visits) - {valid_records})",
            f"{year} 年访问记录数回算失败",
        )
        _assert_zero(
            connection,
            """
            SELECT count(*) FROM visits a
            JOIN visits b
              ON a.did=b.did AND a.chain_segment=b.chain_segment
             AND b.visit_order=a.visit_order+1
            WHERE a.city=b.city
            """,
            f"{year} 年连续同城合并失败",
        )

        connection.execute(
            f"""
            CREATE TABLE edges AS
            SELECT
                concat(
                    s.year::VARCHAR, '-', s.did, '-', s.chain_segment::VARCHAR,
                    '-', s.visit_order::VARCHAR, '-', t.visit_order::VARCHAR
                ) AS edge_id,
                s.year,
                s.did,
                s.chain_segment,
                s.visit_order AS source_visit_order,
                t.visit_order AS target_visit_order,
                s.city AS source_city,
                t.city AS target_city,
                (t.visit_order - s.visit_order)::INTEGER AS edge_level,
                (t.visit_order - s.visit_order - 1)::INTEGER AS skipped_visit_count,
                pow({DECAY_BASE}, t.visit_order - s.visit_order - 1)::DOUBLE AS edge_weight,
                s.entry_time AS source_entry_time,
                s.exit_time AS source_exit_time,
                t.entry_time AS target_entry_time,
                t.exit_time AS target_exit_time,
                date_diff('second', s.exit_time, t.entry_time) / 3600.0 AS gap_hours,
                s.raw_record_count AS source_record_count,
                t.raw_record_count AS target_record_count,
                s.first_source_row_no AS source_first_row_no,
                s.last_source_row_no AS source_last_row_no,
                t.first_source_row_no AS target_first_row_no,
                t.last_source_row_no AS target_last_row_no,
                s.city_visit_no > 1 AS source_is_revisit,
                t.city_visit_no > 1 AS target_is_revisit,
                t.city_visit_no AS target_city_visit_no,
                (t.visit_order - s.visit_order = 1 AND t.city_visit_no > 1) AS is_direct_return_edge,
                s.city = t.city AS is_same_city_endpoint,
                date_diff('second', s.exit_time, t.entry_time) < 0 AS has_time_conflict,
                s.entry_time = t.entry_time AS has_same_start_time,
                '{input_rel}' AS source_file
            FROM visits s
            JOIN visits t
              ON s.did=t.did
             AND s.chain_segment=t.chain_segment
             AND s.visit_order<t.visit_order
            """
        )

        actual_edges = int(_scalar(connection, "SELECT count(*) FROM edges"))
        level_one_edges = int(_scalar(connection, "SELECT count(*) FROM edges WHERE edge_level=1"))
        actual_weight = float(_scalar(connection, "SELECT sum(edge_weight) FROM edges") or 0.0)
        if actual_edges != expected_edges:
            raise RuntimeError(f"{year} 年边数 {actual_edges:,} 与理论值 {expected_edges:,} 不一致。")
        if level_one_edges != expected_level_one:
            raise RuntimeError(
                f"{year} 年一级边数 {level_one_edges:,} 与理论值 {expected_level_one:,} 不一致。"
            )
        if abs(actual_weight - expected_weight) > max(1e-9, expected_weight * 1e-12):
            raise RuntimeError(f"{year} 年权重合计与链级公式不一致。")
        _assert_zero(
            connection,
            """
            SELECT count(*) FROM (
                SELECT year, did, chain_segment, source_visit_order, target_visit_order, count(*) n
                FROM edges GROUP BY ALL HAVING n>1
            )
            """,
            f"{year} 年候选主键重复",
        )
        _assert_zero(
            connection,
            f"""
            SELECT count(*) FROM edges
            WHERE edge_level < 1
               OR skipped_visit_count <> edge_level - 1
               OR abs(edge_weight - pow({DECAY_BASE}, edge_level - 1)) > 1e-15
               OR source_city IS NULL OR target_city IS NULL
               OR source_visit_order >= target_visit_order
            """,
            f"{year} 年边级、权重、方向或端点检查失败",
        )
        _assert_zero(
            connection,
            "SELECT count(*) FROM edges WHERE edge_level=1 AND is_same_city_endpoint",
            f"{year} 年一级边出现同城自环",
        )

        output_query = f"""
            SELECT
                edge_id AS "边记录编号",
                year AS "年份",
                did AS "农机编号",
                chain_segment AS "链段编号",
                source_visit_order AS "起点访问序号",
                target_visit_order AS "终点访问序号",
                source_city AS "起点城市",
                target_city AS "终点城市",
                edge_level AS "边级",
                skipped_visit_count AS "跨越城市数",
                edge_weight AS "衰减权重",
                source_entry_time AS "起点进入时间",
                source_exit_time AS "起点离开时间",
                target_entry_time AS "终点进入时间",
                target_exit_time AS "终点离开时间",
                gap_hours AS "时间间隔小时",
                source_record_count AS "起点原始记录数",
                target_record_count AS "终点原始记录数",
                source_first_row_no AS "起点首行号",
                source_last_row_no AS "起点末行号",
                target_first_row_no AS "终点首行号",
                target_last_row_no AS "终点末行号",
                source_is_revisit AS "起点是否回访",
                target_is_revisit AS "终点是否回访",
                target_city_visit_no AS "终点城市访问序次",
                is_direct_return_edge AS "是否直接回流边",
                is_same_city_endpoint AS "是否同城端点",
                has_time_conflict AS "是否时间冲突",
                has_same_start_time AS "是否同起点时间",
                source_file AS "来源文件"
            FROM edges
            ORDER BY year, did, chain_segment, source_visit_order, target_visit_order
        """
        connection.execute(
            f"COPY ({output_query}) TO '{_sql_path(csv_without_bom)}' "
            "(HEADER, DELIMITER ',', QUOTE '\"', ESCAPE '\"', NULL '')"
        )
        _copy_with_bom(csv_without_bom, csv_with_bom)

        readback_count = int(
            _scalar(
                connection,
                f"SELECT count(*) FROM read_csv_auto('{_sql_path(csv_with_bom)}', header=true, all_varchar=true)",
            )
        )
        if readback_count != actual_edges:
            raise RuntimeError(f"{year} 年输出读回行数不一致。")
        if _read_header(csv_with_bom) != OUTPUT_COLUMNS:
            raise RuntimeError(f"{year} 年输出字段或字段顺序不一致。")

        same_city_edges = int(_scalar(connection, "SELECT count(*) FROM edges WHERE is_same_city_endpoint"))
        target_revisit_edges = int(_scalar(connection, "SELECT count(*) FROM edges WHERE target_is_revisit"))
        direct_return_edges = int(_scalar(connection, "SELECT count(*) FROM edges WHERE is_direct_return_edge"))
        time_conflict_edges = int(_scalar(connection, "SELECT count(*) FROM edges WHERE has_time_conflict"))
        same_start_edges = int(_scalar(connection, "SELECT count(*) FROM edges WHERE has_same_start_time"))
        input_hash = _sha256(input_path)
        output_hash = _sha256(csv_with_bom)

        metrics = [
            ("输入", "输入行数", input_rows, "行", "通过", "与数据字典中的行数一致"),
            ("输入", "农机数", unique_did, "台", "通过", "按did去重"),
            ("输入", "有名称的城市数", city_count, "个", "通过", "保留全部有名称的城市，不限于81城"),
            ("输入", "最早日期", date_min, "日期", "通过", ""),
            ("输入", "最晚日期", date_max, "日期", "通过", ""),
            ("检查", "did为空", missing_did, "行", "通过", "必须为0，否则停止"),
            ("检查", "日期无法读取", invalid_date, "行", "通过", "必须为0，否则停止"),
            ("检查", "开始时间无法读取", invalid_start, "行", "通过", "必须为0，否则停止"),
            ("检查", "结束时间无法读取", invalid_end, "行", "通过", "必须为0，否则停止"),
            ("删除", "城市为空而删除的记录", missing_city, "行", "已记录", "不进入接续数据；明细见城市缺失删除记录.csv"),
            ("检查", "结束时间早于开始时间", reversed_time, "行", "通过", "必须为0，否则停止"),
            ("检查", "date与start_time日期不同", date_time_mismatch, "行", "已记录", "保留原值，不自动修改"),
            ("排序", "同一农机同一开始时间但城市不同", same_start_diff_city_groups, "组", "已记录", "按固定顺序保留"),
            ("排序", "最后需要用原始行号排序的并列记录", exact_sort_tie_groups, "组", "已记录", "不自动去重"),
            ("处理", "删除后保留的记录", valid_records, "行", "通过", "输入行数减去城市为空的记录"),
            ("处理", "农机年度顺序数", chain_segments, "个", "通过", "每台农机每年形成一条城市顺序"),
            ("处理", "合并后的城市访问数", visit_count, "次", "通过", "连续同城合并；离开后再次返回仍保留"),
            ("处理", "合并到其他同城记录中的行数", valid_records - visit_count, "行", "通过", "数量仍可从起点和终点原始记录数查看"),
            ("处理", "单台农机最多访问城市次数", max_visits, "次", "通过", "用于检查数据规模"),
            ("输出", "按公式应有的边数", expected_edges, "行", "通过", "逐台农机按L(L-1)/2计算"),
            ("输出", "实际边数", actual_edges, "行", "通过", "与公式计算结果一致"),
            ("输出", "一级边数", level_one_edges, "行", "通过", "一级边连接相邻的两次城市访问"),
            ("输出", "权重合计", format(actual_weight, ".15g"), "权重", "通过", "与计算公式一致"),
            ("回访", "终点是再次访问城市的边", target_revisit_edges, "行", "已记录", "终点城市此前出现过"),
            ("回访", "直接回流边", direct_return_edges, "行", "已记录", "相邻访问回到此前到过的城市"),
            ("回访", "起点和终点是同一城市的边", same_city_edges, "行", "已记录", "表示离开后又回到同一城市"),
            ("时间", "前后时间重叠的边", time_conflict_edges, "行", "已记录", "时间间隔小于0；目前保留"),
            ("时间", "起点时间相同的边", same_start_edges, "行", "已记录", "目前保留"),
            ("文件核对", "输入文件大小", input_path.stat().st_size, "字节", "通过", ""),
            ("文件核对", "输入文件SHA-256", input_hash, "文件指纹", "通过", "用途见数据字典中的简单说明"),
            ("文件核对", "输出文件大小", csv_with_bom.stat().st_size, "字节", "通过", "文件使用UTF-8-SIG编码"),
            ("文件核对", "输出文件SHA-256", output_hash, "文件指纹", "通过", "用途见数据字典中的简单说明"),
        ]
        for group, metric, value, unit, status, note in metrics:
            _audit_row(
                audit_rows,
                run_id,
                year,
                group,
                metric,
                value,
                unit,
                status,
                note,
                input_rel,
                output_rel,
            )
        return {
            "year": year,
            "input": input_rel,
            "output": output_rel,
            "input_rows": input_rows,
            "valid_records": valid_records,
            "deleted_missing_city": missing_city,
            "chain_segments": chain_segments,
            "visits": visit_count,
            "edges": actual_edges,
            "level_one_edges": level_one_edges,
            "same_city_endpoint_edges": same_city_edges,
            "direct_return_edges": direct_return_edges,
            "time_conflict_edges": time_conflict_edges,
            "output_bytes": csv_with_bom.stat().st_size,
            "input_sha256": input_hash,
            "output_sha256": output_hash,
            "temp_output": str(csv_with_bom),
        }
    finally:
        connection.close()

In [6]:
def _write_audit(path: Path, rows: list[dict[str, Any]]) -> None:
    fieldnames = [
        "运行编号",
        "运行时间",
        "年份",
        "指标组",
        "指标",
        "数值",
        "单位",
        "检查状态",
        "说明",
        "输入文件",
        "输出文件",
    ]
    with path.open("w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def _write_deletion_log(path: Path, rows: list[dict[str, Any]]) -> None:
    """保存被删除的城市为空记录，方便以后查回原始行。"""
    with path.open("w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=DELETION_COLUMNS)
        writer.writeheader()
        writer.writerows(rows)


def build_all_years(
    project_root: Path,
    notebook_path: Path,
    allow_overwrite: bool = False,
) -> dict[str, Any]:
    """生成三年输出；默认拒绝覆盖，允许覆盖时会先自动备份旧输出。"""
    project_root = project_root.resolve()
    inspect_inputs(project_root)
    run_unit_tests()
    output_paths = [(project_root / cfg["output"]).resolve() for cfg in YEAR_CONFIG.values()]
    audit_path = (project_root / AUDIT_RELATIVE_PATH).resolve()
    deletion_path = (project_root / DELETION_RELATIVE_PATH).resolve()
    all_formal_paths = output_paths + [audit_path, deletion_path]
    existing = [path for path in all_formal_paths if path.exists()]
    if existing and not allow_overwrite:
        shown = "\n".join(str(path) for path in existing)
        raise FileExistsError(
            "以下正式输出已存在。为避免覆盖已停止；确认重跑时请将 "
            f"allow_overwrite 设为 True，代码会先备份：\n{shown}"
        )
    for path in all_formal_paths:
        path.parent.mkdir(parents=True, exist_ok=True)
    backup_relative = _prepare_overwrite_backup(project_root, all_formal_paths) if existing else ""
    run_id = datetime.now(ZoneInfo("Asia/Shanghai")).strftime("%Y%m%dT%H%M%S%z")
    audit_rows: list[dict[str, Any]] = []
    deletion_rows: list[dict[str, Any]] = []
    with tempfile.TemporaryDirectory(
        prefix="城市接续_",
        ignore_cleanup_errors=True,
    ) as tmp:
        temp_dir = Path(tmp)
        summaries = [
            _process_year(
                project_root, temp_dir, year, config, run_id, audit_rows, deletion_rows
            )
            for year, config in YEAR_CONFIG.items()
        ]
        expected_deleted_rows = sum(
            config["expected_missing_city"] for config in YEAR_CONFIG.values()
        )
        if len(deletion_rows) != expected_deleted_rows:
            raise RuntimeError(
                f"删除记录应为 {expected_deleted_rows} 行，实际为 {len(deletion_rows)} 行。"
            )
        deletion_temp = temp_dir / deletion_path.name
        _write_deletion_log(deletion_temp, deletion_rows)
        _audit_row(
            audit_rows,
            run_id,
            "全部",
            "删除",
            "城市为空删除记录总数",
            len(deletion_rows),
            "行",
            "通过",
            "明细单独保存在城市缺失删除记录.csv",
            "",
            DELETION_RELATIVE_PATH,
        )
        _audit_row(
            audit_rows,
            run_id,
            "全部",
            "文件核对",
            "删除记录SHA-256",
            _sha256(deletion_temp),
            "文件指纹",
            "通过",
            "用途见数据字典中的简单说明",
            "",
            DELETION_RELATIVE_PATH,
        )
        provenance_files = [
            (
                notebook_path.resolve(),
                "Notebook运行前SHA-256",
                "相当于文件指纹；记录开始运行时的Notebook版本",
            ),
        ]
        for path, metric, note in provenance_files:
            if path.exists():
                _audit_row(
                    audit_rows,
                    run_id,
                    "全部",
                    "文件核对",
                    metric,
                    _sha256(path),
                    "文件指纹",
                    "通过",
                    f"{note}：{str(path.relative_to(project_root)).replace('\\', '/')}",
                    "",
                    AUDIT_RELATIVE_PATH,
                )
        if backup_relative:
            _audit_row(
                audit_rows,
                run_id,
                "全部",
                "覆盖保护",
                "旧输出备份目录",
                backup_relative,
                "路径",
                "通过",
                "重跑前已备份所有既有正式输出",
                "",
                AUDIT_RELATIVE_PATH,
            )
        audit_temp = temp_dir / audit_path.name
        _write_audit(audit_temp, audit_rows)
        for summary, destination in zip(summaries, output_paths, strict=True):
            os.replace(summary["temp_output"], destination)
            summary.pop("temp_output")
        os.replace(deletion_temp, deletion_path)
        os.replace(audit_temp, audit_path)

    result = {
        "run_id": run_id,
        "write_date": WRITE_DATE,
        "decay_base": DECAY_BASE,
        "all_cities_retained": True,
        "maximum_gap_filter": None,
        "backup": backup_relative or None,
        "audit": AUDIT_RELATIVE_PATH,
        "deletion_log": DELETION_RELATIVE_PATH,
        "deleted_rows": len(deletion_rows),
        "years": summaries,
    }
    print(json.dumps(result, ensure_ascii=False, indent=2))
    return result

In [7]:
run_result = build_all_years(
    PROJECT_ROOT,
    NOTEBOOK_PATH,
    allow_overwrite=ALLOW_OVERWRITE,
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{
  "run_id": "20261017T174510+0800",
  "write_date": "2026-10-16",
  "decay_base": 0.5,
  "all_cities_retained": true,
  "maximum_gap_filter": null,
  "backup": "Archive/2026-10-17_174510_城市接续重跑前",
  "audit": "Results/00_数据审计/城市接续审计.csv",
  "deletion_log": "Results/00_数据审计/城市缺失删除记录.csv",
  "deleted_rows": 5,
  "years": [
    {
      "year": 2021,
      "input": "Data/00_原始数据_只读/final_filtered_21_modified.csv",
      "output": "Data/03_有向边数据/2021年农机城市接续.csv",
      "input_rows": 770845,
      "valid_records": 770844,
      "deleted_missing_city": 1,
      "chain_segments": 24676,
      "visits": 61919,
      "edges": 110458,
      "level_one_edges": 37243,
      "same_city_endpoint_edges": 7850,
      "direct_return_edges": 5714,
      "time_conflict_edges": 0,
      "output_bytes": 33991468,
      "input_sha256": "990dc1a3b4406dcd23511d6df83a1925ce60c3fd9f034fce31dde00d93c602f8",
      "output_sha256": "7985254330ed1795fa3d7178d818f8bcfe73f20384b06fe9859ed7fc2a3ce13e"
    },
    {
   

## 检查

In [8]:
audit_path = PROJECT_ROOT / AUDIT_RELATIVE_PATH
deletion_path = PROJECT_ROOT / DELETION_RELATIVE_PATH
if not audit_path.is_file():
    raise FileNotFoundError(f"没有生成检查记录：{audit_path}")
if not deletion_path.is_file():
    raise FileNotFoundError(f"没有生成删除记录：{deletion_path}")

audit_connection = duckdb.connect()
try:
    output_checks = []
    for summary in run_result["years"]:
        output_path = PROJECT_ROOT / summary["output"]
        if not output_path.is_file():
            raise FileNotFoundError(f"没有生成年度数据：{output_path}")
        readback_rows = int(
            audit_connection.execute(
                "SELECT count(*) FROM read_csv_auto(?, header=true, all_varchar=true)",
                [str(output_path)],
            ).fetchone()[0]
        )
        if readback_rows != summary["edges"]:
            raise RuntimeError(
                f"{summary['year']}年文件有 {readback_rows:,} 行，"
                f"但代码记录为 {summary['edges']:,} 行。"
            )
        output_checks.append(
            {
                "年份": summary["year"],
                "输出行数": readback_rows,
                "删除的城市空值记录": summary["deleted_missing_city"],
                "农机年度顺序数": summary["chain_segments"],
                "城市访问数": summary["visits"],
                "一级边数": summary["level_one_edges"],
                "输出SHA-256": summary["output_sha256"],
            }
        )
finally:
    audit_connection.close()

deletion_count = sum(1 for _ in deletion_path.open("r", encoding="utf-8-sig")) - 1
if deletion_count != 5:
    raise RuntimeError(f"删除记录应为5行，实际为{deletion_count}行。")

audit_count = sum(1 for _ in audit_path.open("r", encoding="utf-8-sig")) - 1
if audit_count <= 0:
    raise RuntimeError("检查记录为空。")

print(
    f"检查通过：3个年度数据都能正常读取；删除记录为{deletion_count}行；"
    f"检查记录为{audit_count}行。"
)
output_checks

检查通过：3个年度数据都能正常读取；删除记录为5行；检查记录为100行。


[{'年份': 2021,
  '输出行数': 110458,
  '删除的城市空值记录': 1,
  '农机年度顺序数': 24676,
  '城市访问数': 61919,
  '一级边数': 37243,
  '输出SHA-256': '7985254330ed1795fa3d7178d818f8bcfe73f20384b06fe9859ed7fc2a3ce13e'},
 {'年份': 2022,
  '输出行数': 275055,
  '删除的城市空值记录': 0,
  '农机年度顺序数': 57317,
  '城市访问数': 150753,
  '一级边数': 93436,
  '输出SHA-256': 'ba07c3db48ff7f227dceac4381735802b5971ea53c2cac40acb494099a70e952'},
 {'年份': 2023,
  '输出行数': 500521,
  '删除的城市空值记录': 4,
  '农机年度顺序数': 83329,
  '城市访问数': 237110,
  '一级边数': 153781,
  '输出SHA-256': 'a5c29cb0d8b16908934735c686ed98cacefa44ba4bd2136a472e463140ad5662'}]

## 输出

In [9]:
result_summary = [
    {
        "年份": item["year"],
        "输入行数": item["input_rows"],
        "删除行数": item["deleted_missing_city"],
        "删除后保留行数": item["valid_records"],
        "城市访问数": item["visits"],
        "接续边数": item["edges"],
        "直接回流边数": item["direct_return_edges"],
        "同城端点边数": item["same_city_endpoint_edges"],
        "时间重叠边数": item["time_conflict_edges"],
        "输出文件": item["output"],
    }
    for item in run_result["years"]
]
result_summary

[{'年份': 2021,
  '输入行数': 770845,
  '删除行数': 1,
  '删除后保留行数': 770844,
  '城市访问数': 61919,
  '接续边数': 110458,
  '直接回流边数': 5714,
  '同城端点边数': 7850,
  '时间重叠边数': 0,
  '输出文件': 'Data/03_有向边数据/2021年农机城市接续.csv'},
 {'年份': 2022,
  '输入行数': 2475805,
  '删除行数': 0,
  '删除后保留行数': 2475805,
  '城市访问数': 150753,
  '接续边数': 275055,
  '直接回流边数': 16677,
  '同城端点边数': 25744,
  '时间重叠边数': 7,
  '输出文件': 'Data/03_有向边数据/2022年农机城市接续.csv'},
 {'年份': 2023,
  '输入行数': 3976426,
  '删除行数': 4,
  '删除后保留行数': 3976422,
  '城市访问数': 237110,
  '接续边数': 500521,
  '直接回流边数': 28912,
  '同城端点边数': 42831,
  '时间重叠边数': 25,
  '输出文件': 'Data/03_有向边数据/2023年农机城市接续.csv'}]

三份年度数据位于 `Data/03_有向边数据`。5行删除明细位于 `Results/00_数据审计/城市缺失删除记录.csv`，运行检查记录位于 `Results/00_数据审计/城市接续审计.csv`。先检查这些文件，再进行下一步。